In [7]:
from __future__ import division
import math
import numpy as np

%matplotlib inline

import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']  # Colors for plotting

n_colors = len(colors)

In [8]:
# A) Initialisation des variables  #####################################################################

# 1. Constantes physiques (NIST)
h    = 6.626068e-34;  # [J.S]      Planck constant
heV  = 4.135667e-15;  # [eV.s]     Planck constant
hb   = h/(2*math.pi);      # [J.s]      h barre
hbeV = heV/(2*math.pi);    # [eV.s]     h barre
me   = 9.109381e-31;  # [kg]       electron mass
q    = 1.6021764e-19; # [C]        electron charge
c    = 299792458;     # [m.s^(-1)] speed of light in vacuum

# 2. Propriété de la particule étudiée
m = me;           # [kg] masse (me = masse de l'électron)

# 3. Propriétés de l'espace étudié (puits, maille,...)
xMIN = -1e-8 # [m] extension maximale vers la gauche
xMAX = -xMIN # [m] extension maximale vers la droite
xNum = 301   #  [-] nombre de valeurs discrétisées de x (résolution) doit être impair


# 4. Niveaux énergétiques et fonctions d'onde
vectEn   = np.arange(0,10,1) # nombre niveaux énergétiques sélectionnés pour le plot
vectPSIn = np.arange(0,5,1)  # nombre fonctions d'ondes sélectionnées pour le plot

# 5 Définitions de constantes internes
dx    = (xMAX-xMIN)/(xNum-1) # [m]
xArr  = np.linspace(xMIN, xMAX, xNum-1)
xOnes = [1.0] * (xNum-1)


In [9]:
# B) Définition du potentiel ####################################################################################

def Potentiel(idxPot, nPuits, Profondeur, Largeur, Distance, ax1):
    """
    Calculate and plot a potential based on the type (rectangular, Coulomb, or quartic).

    Parameters:
    -----------
    idxPot : int
        Index to choose the type of potential (1 for rectangular, 2 for Coulomb, 4 for quartic).
    nPuits : int
        Number of potential wells.
    Profondeur : float
        Depth of the potential wells (in eV).
    Largeur : float
        Width of the potential wells (in nm).
    Distance : float
        Distance between the potential wells (in nm).
    ax1 : matplotlib axis
        Axis on which the potential will be plotted.

    Returns:
    --------
    Pot : list
        List of potential values calculated for each point on the x-axis.
    """

    # Convert units: Profondeur in eV, Largeur and Distance in meters
    Profondeur = Profondeur * q
    Largeur = Largeur * 1e-9
    Distance = Distance * 1e-9

    # Initialize the list of potential values
    Pot = []

    # Rectangular potential
    if idxPot == 1:
        for xx in xArr:
            inWell = False

            # Even number of wells
            if nPuits % 2 == 0:
                for ip in range(-int(nPuits/2), int(nPuits/2)):
                    if xx >= (Distance/2 + ip*(Largeur+Distance)) and xx <= (Distance/2 + Largeur + ip*(Largeur+Distance)):
                        inWell = True

            # Odd number of wells
            else:
                for ip in range(-int((nPuits-1)/2), int((nPuits+1)/2)):
                    if xx >= (-Largeur/2 + ip*(Largeur+Distance)) and xx <= (Largeur/2 + ip*(Largeur+Distance)):
                        inWell = True

            Pot.append(-Profondeur if inWell else 0)

    # Quartic potential
    elif idxPot == 4:
        for xx in xArr:
            potval = 0

            if nPuits == 1:
                potval = Profondeur * (xx/Largeur)**4
            else:
                raise NotImplementedError("Quartic potential is only implemented for a single well (nPuits=1).")

            Pot.append(max(potval, -Profondeur))

    # Coulomb potential
    else:
        for xx in xArr:
            potval = 0

            if nPuits % 2 == 0:
                for ip in range(-int(nPuits/2), int(nPuits/2)):
                    potval += -q/abs((xx-ip*(Largeur+Distance)-(Largeur+Distance)/2)/(Largeur/20))

            else:
                for ip in range(-int((nPuits-1)/2), int((nPuits+1)/2)):
                    potval += -q/abs((xx-ip*(Largeur+Distance))/(Largeur/20))

            Pot.append(max(potval, -Profondeur))

    # Plot the potential
    ax1.plot(xArr*1e9, np.array(Pot)/q, color='black', label='Potentiel')
    ax1.set_xlabel("Distance (nm)")
    ax1.set_ylabel("Potentiel (eV)")
    ax1.grid(True)

    return Pot

In [10]:
# C) Calcul des états propres par différences finies ########################################################

def Schroedinger(Pot, nEn, ax1, ax2):
    # 1. Define diagonals of the kinetic energy matrix T
    diag2 = [1.0] * int(xNum-2)
    diag3 = [1.0] * int(xNum-3)

    # Kinetic energy matrix T
    matDiff = -2*np.diag(diag2, k=0) + np.diag(diag3, k=1) + np.diag(diag3, k=-1)
    matKin = -(hb**2)/(2*m) * matDiff/(dx**2)

    # Potential energy matrix
    matPot = np.diag(Pot[1:int(xNum-1)], k=0)

    # 2. Hamiltonian H = T + V
    H = matKin + matPot

    # 3. Solve the eigenvalue equation H Psi = E Psi
    E, Psi = np.linalg.eig(H)

    # Sort eigenvalues and eigenvectors
    E_sorted = np.sort(E)
    sorted_Psi = Psi[:, np.argsort(E)]

    # Plot energy levels
    for ii in range(nEn):
        ax1.axhline(y=E_sorted[ii]/q, color=colors[ii % len(colors)], linestyle='--', label=f'E{ii}')
        print(f"E{ii} = {E_sorted[ii]/q} eV")

    ax1.legend()

    # Wave functions
    Psi_new = np.insert(sorted_Psi, 0, 0, axis=1)
    Psi_new = np.insert(Psi_new, xNum-1, 0, axis=1)

    # Plot wave functions
    for ii in range(nEn):
        ax2.plot(xArr[1:None]*1e9, Psi_new[:, ii+1], color=colors[ii % len(colors)], label=f'Psi{ii}')

    ax2.set_xlabel("Distance (nm)")
    ax2.set_ylabel("Fonction d'onde")
    ax2.grid(True)
    ax2.legend()

In [11]:
def func(potentiel, profondeur, largeur, nPuits, distance, nEn):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Potentiel et Fonctions d'onde")

    pot = Potentiel(potentiel, nPuits, profondeur, largeur, distance, ax1)
    Schroedinger(pot, nEn, ax1, ax2)

    plt.tight_layout()
    plt.show()

In [12]:
# Widgets
potentiel = widgets.Dropdown(
    options=[
        ('Rectangulaire', 1),
        ('Coulombien', 2),
        ('Quartique', 4),
    ],
    value=1,
    description='Potentiel:',
)

profondeur = widgets.FloatSlider(value=0.5, min=0.0001, max=2, step=0.1, description='Profondeur (eV):')
largeur = widgets.FloatSlider(value=2, min=0.1, max=10, step=0.5, description='Largeur (nm):')
distance = widgets.FloatSlider(value=2, min=0, max=10, step=0.5, description='Distance (nm):')
nPuits = widgets.IntSlider(value=1, min=1, max=4, step=1, description='# puits:')
nEn = widgets.IntSlider(value=4, min=1, max=10, step=1, description='# états:')

# Use `interact` to link widgets to the function
interact(
    func,
    potentiel=potentiel,
    profondeur=profondeur,
    largeur=largeur,
    nPuits=nPuits,
    distance=distance,
    nEn=nEn
);

interactive(children=(Dropdown(description='Potentiel:', options=(('Rectangulaire', 1), ('Coulombien', 2), ('Q…